In [24]:
import pandas as pd
import numpy as np
import glob

POLICY_PATH = "/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/policy_model_data.parquet"
WIN_PATH     = "/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/processed/win_model_data.parquet"
# === Load all parquet files in directory ===
files = glob.glob("/Users/aleksandar.stojanovic/Downloads/ai_system/ml_scripts/data/raw/*.parquet")
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)


# === Helper function to encode board state ===
def encode_board_state(board):
    symbol_map = {'X': 1, 'O': -1, ' ': 0}
    return [symbol_map.get(cell, 0) for cell in board]

# === Encode legal moves as binary vector ===
def encode_legal_moves(legal_moves):
    binary = [0] * 9
    for i in legal_moves:
        if 0 <= i < 9:
            binary[i] = 1
    return binary

# === Common processing: apply encoding ===
df["boardEncoded"] = df["boardState"].apply(encode_board_state)
df["playerEncoded"] = df["currentPlayer"].map({'X': 1, 'O': -1})
df["legalMovesEncoded"] = df["legalMoves"].apply(encode_legal_moves)

# === Outcome encoding for win model ===
outcome_map = {'WIN_X': 0, 'WIN_O': 1, 'DRAW': 2}
df["outcomeEncoded"] = df["outcome"].map(outcome_map)

# === Filter 1: Policy model (only rows with strong label present) ===
policy_df = df[df["bestMoveStrong"].notnull()].copy()

# Features: board, player, (optionally legal moves)
policy_df["features"] = policy_df.apply(
    lambda row: row["boardEncoded"] + [row["playerEncoded"]],
    axis=1
)
# Optional: include legal moves if desired
# + row["legalMovesEncoded"]

# Target
policy_df["target"] = policy_df["bestMoveStrong"].astype(int)

# Save
policy_df[["features", "target"]].to_parquet(POLICY_PATH, index=False)

# === Filter 2: Win model (any row with known outcome label) ===
win_df = df[df["outcome"].isin(outcome_map.keys())].copy()

# Features: board, player, move number
win_df["features"] = win_df.apply(
    lambda row: row["boardEncoded"] + [row["playerEncoded"], row["moveNumber"]],
    axis=1
)
win_df["target"] = win_df["outcomeEncoded"].astype(int)

# Save
win_df[["features", "target"]].to_parquet(WIN_PATH, index=False)

print("  Preprocessing complete. Saved:")
print("- policy_model_data.parquet")
print("- win_model_data.parquet")


  Preprocessing complete. Saved:
- policy_model_data.parquet
- win_model_data.parquet
